# Netflix catalogue exploration

This notebook only looks. Training, the leakage audit, and the API live in `src/` and `scripts/`.
The numbers below are computed by those modules from `data/raw/netflix_titles.csv`.

In [ ]:
from src.data.clean import clean_catalogue
from src.data.load import load_raw_csv

raw = load_raw_csv()
cleaned = clean_catalogue(raw)
print("rows", len(cleaned))
print(cleaned["type"].value_counts())
print("synthesized", cleaned.attrs.get("synthesized_columns"))

## The duration shortcut

Every movie duration is in minutes and every TV show duration is in seasons.
A model that sees `duration_unit` is not predicting anything.

In [ ]:
print(cleaned.groupby(["duration_unit", "type"]).size().unstack(fill_value=0))
print("director missing by type")
print(cleaned.groupby([cleaned["has_director"].eq(0), "type"]).size().unstack(fill_value=0))

## What the honest model is allowed to use

Title text, country, rating, years, and counts. Not duration, not genre names, not whether a director credit was filled in.
This extract has no synopsis, so description length is not a feature.
The written audit is `reports/leakage_audit.md`.

In [ ]:
from pathlib import Path

from src.features.engineer import honest_spec

spec = honest_spec(cleaned)
print("text", spec.text_columns)
print("categorical", spec.categorical_columns)
print("numeric", spec.numeric_columns)
audit = Path("reports/leakage_audit.md")
print(audit.read_text(encoding="utf-8").split("## Verdicts")[0][:1500])